# 04 — Evaluation: P1 / P2 / ablations / Qwen hallucination (OFFLINE, Blackwell)

| attach as input | produces |
|---|---|
| `behaviorsense-code`, `behavioursense-WW`, `behaviorsense-adl-shards`, `behaviorsense-fall-shards`, `behaviorsense-runs` | `results/evaluation.md` + `results/hallucination_qwen.md` in the output (publish as `behaviorsense-results`) |
| Kaggle Model: **Qwen2.5-7B-Instruct** (attach via Add Input -> Models; no upload needed) | |

Protocols: **P1** subject-disjoint validation (same split seed as training, so these
windows influenced training only via early stopping); **P2** leave-one-dataset-out on the
fall sources; per-stream and combination **ablations**; the **hallucination table** with
the stub anchors from `results/hallucination.md` giving the 0%/injection-rate calibration
that makes the Qwen numbers interpretable.

Two things changed after the first complete run of this notebook:

1. **Every table is written to `results/evaluation.md`.** The previous bundling cell
   globbed `results/*.md` and found only the file a subprocess had written, so nine hours
   of P1 / per-class / calibration / P2 / ablation output existed solely as session
   scrollback. The session log is not a record.
2. **The hallucination table has three arms, two denominators and a confidence interval.**
   Run 1 scored the free arm as 245 claims emitted / 0 scorable / `nan%`, and the
   constrained arm at 39.9%. Both were artefacts of a prompt that never named the output
   fields: the free model could not guess the envelope, and the constrained model
   mis-assigned values. With the mapping stated, run 2 gave 8.2% / 6.5% and zero schema
   rejections in either arm — so most of that 39.9% was our prompt.
3. **Both arms are now pinned to greedy decoding.** Runs 2 and 3 disagreed: the free arm was
   bit-identical (509 claims, 476 faithful, twice) while the constrained arm moved 498/457 →
   504/452 and the verdict flipped from p=0.29 to p=0.03. The free path passed
   `do_sample=False`; the constrained path passed only `max_new_tokens` to `outlines` and
   inherited Qwen's `generation_config` (`do_sample=True, temperature=0.7`). One arm was
   greedy, the other sampled, so the gap measured temperature rather than grammar.
   `force_greedy()` pins both — **this is the first decoding-matched run, and the earlier
   constrained numbers should not be quoted.**

In [ ]:
# Resolve every attached asset by CONTENT, not by dataset name.
#
# Kaggle mount paths are not predictable from here, and three separate things vary:
#   - notebook 00 emits two folders, which can be published as ONE dataset or two
#     (observed: a single "behavioursense-WW" holding both wheels/ and weights/)
#   - the dataset title is free text, and "behaviour" vs "behavior" both occur
#   - Save Version nests the working directory inside the dataset, so files end up at
#     <mount>/kaggle/working/... rather than <mount>/...
#
# Guessing the name has already cost one session, and notebook 01's carry-forward bug
# showed how the failure presents: a wrong path reads as "nothing attached", the run
# continues, and work is skipped or destroyed rather than failing loudly.
#
# So identify each asset by a file only it has. A directory holding *.whl is the wheel
# cache no matter what the dataset is called.
import pathlib
from itertools import islice

INPUT = pathlib.Path("/kaggle/input")
def attached_mounts():
    # Datasets do NOT sit directly under /kaggle/input. They mount at
    # /kaggle/input/datasets/<owner>/<name>/, and competitions at
    # /kaggle/input/competitions/<name>/. Listing INPUT.iterdir() therefore always
    # reports ['competitions', 'datasets'] whatever is attached - which is what the
    # "code: nothing matches ..." failure printed, telling us nothing about whether
    # the code dataset was attached. Descend to the level that names real mounts, and
    # report what each one CONTAINS, since a dataset can be attached and still be
    # missing the directory the notebook needs.
    out = []
    for container in ("datasets", "competitions"):
        base = INPUT / container
        if not base.is_dir():
            continue
        for owner in sorted(base.iterdir()):
            kids = sorted(owner.iterdir()) if owner.is_dir() else []
            if kids and all(k.is_dir() for k in kids[:1]) and container == "datasets":
                for ds in kids:
                    # islice, NOT sorted(...)[:6]. `sorted()` materialises the whole listing
                    # first, and on Kaggle's FUSE mount a slug whose files sit at its root -
                    # `toyota-smarthome-skeleton-v1-2` holds 16,115 - makes that a full
                    # network directory read per mount. Eleven mounts of that shape is most
                    # of the 14 minutes this cell took on the first Toyota run. Six names
                    # are all this diagnostic needs, so stop after six.
                    top = sorted(islice((q.name for q in ds.iterdir()), 6)) \
                        if ds.is_dir() else []
                    out.append(f"{ds.name} (top level: {top})")
            else:
                out.append(owner.name)
    # Fall back to the flat layout so this keeps working if Kaggle changes the mount
    # shape back, rather than reporting nothing at all.
    return out or sorted(p.name for p in INPUT.iterdir())

ATTACHED = attached_mounts() if INPUT.is_dir() else []

# MOUNT-INDEXED SEARCH, and why two earlier fixes were not enough.
#
# `INPUT.glob("**/x")` walks every directory under /kaggle/input - 93 minutes with the
# Toyota corpus mounted, notebook 06 measured, and 349 s for the single
# `**/src/behaviorsense/__init__.py` probe in notebook 07's second run. The first "fix",
# FIXED-DEPTH globs like `datasets/*/*/*/rtmo-l.onnx`, was depth-bounded but not
# COST-bounded: to match at depth 3 pathlib scandirs EVERY slug child directory, including
# `toyota-smarthome-skeleton-v1-2`'s 16,115-file root and MSMT17's 65,242 crops.
#
# The mounts are KNOWN at depth 2 (`datasets/<owner>/<slug>/`), so enumerate them once and
# resolve everything else with is_file() stats - one metadata call per mount per candidate,
# never a sibling-directory listing.
_SLUGS = (sorted((INPUT / "datasets").glob("*/*"))
          + sorted((INPUT / "competitions").glob("*")))

def find_fast(tail, what, required=True):
    # Known staging prefixes, each costing one stat per mount:
    #   ""                      files at the slug root
    #   EmotionSense-Extended/  the code dataset was created by zipping the repo FOLDER,
    #                           so everything sits one level below the slug
    #   kaggle/working/         Save Version nests the working directory
    # The prefixes apply to MULTI-COMPONENT tails too. They used to be tried only for bare
    # filenames, which quietly sent `src/behaviorsense/__init__.py` - the one probe every
    # notebook makes - down the deep-search path it was written to avoid.
    # `weights/` is additionally tried for a bare filename, the staged weights layout.
    prefixes = ("", "EmotionSense-Extended/", "kaggle/working/")
    mids = ("",) if "/" in tail else ("", "weights/")
    for t in [p + m + tail for p in prefixes for m in mids]:
        hits = [s / t for s in _SLUGS if (s / t).is_file()]
        if hits:
            return hits[0]
    hits = sorted(INPUT.glob(f"**/{tail}"))   # last resort: unusual layout, slow, once
    if hits:
        print(f"  {what:<9} found only by deep search ({tail}) - layout is unusual")
        return hits[0]
    if required:
        raise AssertionError(
            f"{what}: nothing matches {tail!r} in any mount. Attached: {ATTACHED}")
    print(f"  {what:<9} ABSENT (optional)")
    return None

def find_asset(pattern, what, required=True):
    # Callers pass a `**/...` pattern. The leading `**/` is stripped and the mount-indexed
    # search runs first, so every existing call site gets the speed-up unchanged.
    tail = pattern[3:] if pattern.startswith("**/") else pattern
    return find_fast(tail, what, required=required)

def find_dir(subdir, pattern, roots=None):
    # "Which mount holds the most files matching this pattern in this subdirectory?" - one
    # scandir of ONE named directory per mount, never a recursive walk. Used for corpora
    # (Toyota's mp4/ and Videos_mp4/) where the answer is a directory, not a file.
    from fnmatch import fnmatch
    import os
    best, best_n = None, 0
    for root in (roots if roots is not None else _SLUGS):
        base = root / subdir if subdir else root
        if not base.is_dir():
            continue
        n = sum(1 for e in os.scandir(base) if e.is_file() and fnmatch(e.name, pattern))
        if n > best_n:
            best, best_n = base, n
    return best, best_n

def find_charades_csv():
    # The charades-480p dataset nests the CSV one level down (its root holds
    # Charades_annotations/ and Charades_v1_480/), and a `**` glob for it walks the code
    # dataset's MSMT17 copy - minutes for one file. Probe the known shapes per mount;
    # only a genuinely unknown layout falls through to the deep search.
    for slug in _SLUGS:
        if "charades" not in slug.name.lower():
            continue
        for base in (slug, slug / "Charades_annotations", slug / "Charades_v1_480",
                     slug / "kaggle" / "working"):
            p = base / "Charades_v1_train.csv"
            if p.is_file():
                return [p]
    return sorted(INPUT.glob("**/Charades_v1_train.csv"))

def find_wheel_dir():
    # "The directory containing *.whl" is not specific enough: /kaggle/input also holds
    # attached COMPETITIONS, and at least one (arc-prize-2026) ships its own wheels. The
    # first sorted hit was that competition's, and the offline install then failed on a
    # cache that simply does not contain torch. Score candidate directories by how many
    # of OUR packages they hold and take the best.
    MARKERS = {"torch", "rtmlib", "onnxruntime-gpu", "nvidia-cudnn-cu12", "triton"}
    dirs = {}
    for slug in _SLUGS:
        for wdir in (slug / "wheels", slug / "kaggle" / "working" / "wheels"):
            if not wdir.is_dir():
                continue
            for w in wdir.glob("*.whl"):
                dirs.setdefault(wdir, set()).add(
                    w.name.split("-")[0].lower().replace("_", "-"))
        if dirs:
            break
    if not dirs:
        for w in INPUT.glob("**/*.whl"):        # last resort
            dirs.setdefault(w.parent, set()).add(
                w.name.split("-")[0].lower().replace("_", "-"))
    if not dirs:
        raise AssertionError(f"no *.whl anywhere under /kaggle/input. Attached: {ATTACHED}")
    best, hits = max(dirs.items(), key=lambda kv: len(kv[1] & MARKERS))
    if not (hits & MARKERS):
        raise AssertionError(
            f"found {len(dirs)} wheel director(ies) but none holds any of {sorted(MARKERS)} "
            f"- the staged cache from notebook 00 is not attached. Candidates: "
            f"{[str(d) for d in dirs]}")
    return best

WHEELS  = find_wheel_dir()
WEIGHTS = find_asset("**/rtmo-l.onnx", "weights").parent
SRC     = find_asset("**/src/behaviorsense/__init__.py", "code").parent.parent
CODE    = SRC.parent
SCRIPTS = CODE / "scripts"

CONFIGS = CODE / "configs"

# sys.path belongs HERE, in the cell that resolves SRC, and unconditionally.
#
# It used to live at the end of the wheel-install cell. Notebook 04 put it outside that
# cell's `if not sm120_ok()` branch and worked; notebook 03 never had it at all and worked
# anyway, because every heavy step there is a subprocess launched with PYTHONPATH set. Then
# `is_real_artifact` was added to notebook 03's shard resolver - the first in-process import
# of `behaviorsense` in that notebook - and the next run died at cell 4 with
# `ModuleNotFoundError: No module named 'behaviorsense'`, six minutes in, one cell after
# PREFLIGHT PASSED. A path set up as a side effect of an unrelated, conditional cell is a
# dependency nobody can see.
import sys
sys.path.insert(0, str(SCRIPTS))
sys.path.insert(0, str(SRC))

for _label, _path in (("wheels", WHEELS), ("weights", WEIGHTS), ("code", CODE)):
    print(f"  {_label:<8} {_path}")
print(f"  attached  {ATTACHED}")
print("NOTE: if you restart the kernel below, re-run from THIS cell - these names "
      "are what every later cell uses.")

CONTRACT = [
    ("scripts/kaggle_smoke_test.py", "--profile",      "notebook 03 preflight"),
    ("scripts/train_adl.py",         "--stop-after",   "resume guard (notebook 03)"),
    ("scripts/train_adl.py", "--tau-train",
     "notebook 03 passes --sampler/--tau-train; a snapshot predating them exits 2 from "
     "argparse, which notebook 03 reports as a failed stream rather than stale code"),
    ("src/behaviorsense/data/skeleton_dataset.py", "def load_subject_map",
     "video-id -> Charades actor-id remap for a person-disjoint P1 split (notebooks "
     "03/04). A stale snapshot silently reverts P1 to video-disjoint - same person in "
     "train and val - while printing numbers that look identical"),
    ("scripts/train_fall.py", "pos_rate > 0.5 and args.focal_alpha > 0.5",
     "refuses focal alpha that up-weights the majority; a snapshot without it trains "
     "the fall head on 82% positives and reports a plausible but meaningless AUPRC"),
    ("scripts/prepare_skeletons.py", "def assign_slots",
     "slot tracking + windowing (notebooks 01/02)"),
    ("scripts/prepare_skeletons.py", "with_starts",
     "fall labelling by true frame position (notebook 02); index-derived position "
     "mislabels the descent whenever a window is dropped"),
    ("scripts/prepare_skeletons.py", "def le2i_fall_frames",
     "Le2i has no fall/ADL marker in any path component; without this its ~192 fall "
     "clips land in the negatives (notebook 02)"),
    ("scripts/prepare_skeletons.py", "def label_fall_windows",
     "shared fall labelling: exact interval for Le2i, positional fallback elsewhere"),
    ("src/behaviorsense/data/skeleton_dataset.py", "keep_root_motion",
     "falls are unlearnable without it"),
    ("src/behaviorsense/models/ensemble.py", "def per_stream_logits", "notebook 04 ablation"),
    ("src/behaviorsense/models/ensemble.py", "def flip_windows",
     "test-time flip augmentation (notebook 04 levers cell). A stale snapshot would accept "
     "`clf.tta = True` as a new attribute and silently do no TTA, reporting the "
     "unaugmented number as if it were augmented"),
    ("src/behaviorsense/models/stgcnpp.py", "parents.setdefault",
     "flip-equivariant bone stream; without it half the ensemble trains on sign noise. The "
     "token tracked the local name `parent` and broke when the function grew an explicit "
     "`parents` argument for Toyota's 15-node tree - a rename silently disarming a staleness "
     "guard is exactly what this list exists to catch, so it caught itself"),
    ("src/behaviorsense/kaggle_artifacts.py", "def find_run_dir",
     "one rule for 'is this real session output or a dev leftover'; four call sites "
     "learned it separately and the fourth was missed"),
    ("src/behaviorsense/agents/reasoning/reporter.py", "def force_greedy",
     "pins BOTH decoding arms to greedy (notebook 04). Without it the constrained arm "
     "inherits Qwen's generation_config (do_sample=True, temperature=0.7) while the free "
     "arm is greedy, so the comparison measures temperature instead of grammar - the "
     "constrained rate moved 8.2% -> 10.3% between two runs of identical code"),
    ("src/behaviorsense/agents/reasoning/reporter.py", "def _chat_text",
     "both arms send the SAME templated text (notebook 04). The constrained path used to "
     "hand outlines the raw prompt, so one arm got a Qwen chat turn and the other a naked "
     "instruction block - a second confound on top of the sampling one"),
    ("src/behaviorsense/agents/reasoning/reporter.py", "def repair_claim",
     "format-only claim repair + maxItems bound to max_claims (notebook 04). Without "
     "it the unconstrained arm scores 245 emitted / 0 scorable / nan%, which measures "
     "JSON compliance rather than faithfulness"),
    ("scripts/eval_hallucination.py", "unusable_rate",
     "three-arm hallucination table with a denominator over EMITTED claims; a stale "
     "snapshot silently reports the two-arm nan% version"),
    ("src/behaviorsense/eval/activity_eval.py", "def logit_adjust",
     "notebook 04's P1 cell imports MIN_SUPPORT and scores() from here, so a stale "
     "snapshot fails with ImportError at cell 3; also carries the post-hoc accuracy "
     "levers scripts/rescore_p1.py replays off the saved val logits"),
    ("src/behaviorsense/video.py", "def child_env",
     "notebook 05's /video endpoint decodes uploads in a CHILD process (ffmpeg raises SIGSEGV "
     "on malformed streams and a signal is not catchable, so without the boundary one bad "
     "upload kills the kernel, the tunnel and the demo together). `child_env` is what puts "
     "behaviorsense on that child's PYTHONPATH - sys.path does not cross a process boundary, "
     "and a snapshot without it 422s EVERY upload with \"No module named 'behaviorsense'\""),
    ("src/behaviorsense/pipeline.py", "def frames_to_windows", "Agent 1 -> Agent 2 seam"),
    ("configs/taxonomy.yaml",        None,             "class map (notebook 01)"),
]
_stale = []
for _rel, _token, _why in CONTRACT:
    _p = CODE / _rel
    if not _p.is_file():
        _stale.append(f"{_rel} is MISSING ({_why})")
        continue
    if _token and _token not in _p.read_text(encoding="utf-8", errors="ignore"):
        _stale.append(f"{_rel} lacks {_token!r} ({_why})")
if _stale:
    raise AssertionError(
        "The attached code dataset is OLDER than these notebooks:\n  - "
        + "\n  - ".join(_stale)
        + f"\n\nMounted: {CODE}\nRe-upload from your checkout, then restart this notebook:"
          "\n  kaggle datasets version -p . --dir-mode zip -m \"sync\""
    )
print(f"  contract {len(CONTRACT)}/{len(CONTRACT)} - mounted code is current")

In [ ]:
import subprocess, sys
def sm120_ok():
    try:
        import torch
        return torch.cuda.is_available() and "sm_120" in torch.cuda.get_arch_list()
    except Exception:
        return False
if not sm120_ok():
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index",
                    "--find-links", str(WHEELS),
                    "torch", "numpy", "pydantic", "PyYAML", "Pillow"], check=True)
    print("RESTART the kernel, then continue from the next cell.")
subprocess.run([sys.executable, "-m", "pip", "install", "--no-index",
                "--find-links", str(WHEELS),
                "transformers", "accelerate", "outlines", "safetensors"], check=True)
# sys.path is set by the resolver cell now, unconditionally, for every offline notebook.
# It used to be set here, which made an import path depend on a wheel-install cell.

In [ ]:
# P1 - subject-disjoint mean-class accuracy / macro-F1, per stream and ensemble.
#
# val_frac=0.2 and seed=0 are train_adl.py's OWN defaults, so this is exactly the set the
# streams were early-stopped against - not a re-split. It previously used val_frac=0.15,
# which `split_by_subject` makes a strict SUBSET (it shuffles subjects by seed and takes a
# prefix), so there was never leakage - but the comment claimed to "reproduce the
# training-time val split" and did not, and the subset threw away a fifth of the evidence
# for nothing. These windows influenced training only through early stopping and best.pt
# selection, which is what makes every number here validation-selected rather than held-out.
#
# Eval windows are built from the RAW shard arrays with normalise() only. Two traps this
# avoids: ds[i] returns [C,T,V,M] (already permuted for the model) whereas
# EnsembleClassifier.logits() wants the dataset layout [N,T,M,17,3]; and ds[i] would also
# apply temporal_resample jitter. Augmentation is already off (augment_cfg=None ->
# AugmentConfig(enabled=False)), but building eval inputs explicitly is what makes that
# guarantee visible instead of assumed.
import numpy as np
from behaviorsense.kaggle_artifacts import is_real_artifact
from behaviorsense.data.skeleton_dataset import (SkeletonWindowDataset, split_by_subject,
                                                 normalise, N_CLASSES)
from behaviorsense.models.ensemble import EnsembleClassifier
from behaviorsense.agents.activity import CLASS_NAMES
# MIN_SUPPORT and scores() live in the library so this cell, scripts/rescore_p1.py and the
# tests share ONE definition. They used to be inline here, which meant any test of the
# metric validated its own copy - the pattern behind three notebook-04 defects.
from behaviorsense.eval.activity_eval import MIN_SUPPORT, scores

def shard_paths(*keywords):
    # Match on ANY path component, not the top-level mount name: attaching by URL nests
    # the dataset under /kaggle/input/datasets/<owner>/<name>/, so iterating the top level
    # finds only 'datasets' and returns nothing. Cost notebook 03 a session.
    out = []
    for q in INPUT.glob("**/*.npz"):
        if not is_real_artifact(q):
            continue          # fixture, or a leftover inside a mounted code checkout
        rel = [part.lower().replace("behaviour", "behavior") for part in q.parts]
        if any(k in part for part in rel for k in keywords):
            out.append(str(q))
    return sorted(out)

# find_run_dir picks the directory that actually holds adl_<stream>/ subdirectories,
# and skips anything inside a mounted code checkout. The previous rule took the first
# last.pt in sorted order, which selected behaviorsense-code/.../runs ("code" sorts
# before "runs") - a directory holding adl/ and fall/, not adl_joint/. Notebook 04 then
# died reporting a missing checkpoint when the real fault was the wrong directory.
from behaviorsense.kaggle_artifacts import find_run_dir

def run_dir():
    return str(find_run_dir(INPUT))

ADL = shard_paths("adl")
assert ADL, f"no ADL shards found. Attached: {ATTACHED}"
ds = SkeletonWindowDataset([*map(str, ADL)])

# The shards store VIDEO ids as `subjects` - docs/07 said "Charades subject ids do not
# exist publicly", and that was wrong: Charades_v1_train.csv has a `subject` column (267
# actors, ~30 videos each). A video-id split puts nearly every actor on both sides, so
# "subject-disjoint" P1 was video-disjoint: same person, same home, train and val. When
# the CSV is attached (charades-480p carries it), the split is remapped to ACTOR ids and
# P1 becomes person-disjoint. MUST match how the checkpoints were trained - the split
# mode is printed and recorded next to the numbers, because a person-disjoint eval of
# video-disjoint checkpoints reports leakage-free numbers for a leaky model.
from behaviorsense.data.skeleton_dataset import load_subject_map, remap_subjects
_csv = find_charades_csv()
split_subjects, SPLIT_MODE = ds.subjects, "video-id (proxy)"
if _csv:
    split_subjects, _cov = remap_subjects(ds.subjects, load_subject_map(_csv[0]))
    if _cov >= 0.5:
        SPLIT_MODE = f"actor-id ({len(set(split_subjects))} actors, {_cov:.0%} mapped)"
    else:
        split_subjects = ds.subjects
        print(f"subject CSV found but covers only {_cov:.1%} of windows - keeping video ids")

# REFUSE the leaky protocol unless it is asked for explicitly.
#
# This used to fall back to video ids with one printed line, and that is exactly what
# happened: a session ran without `charades-480p` attached, printed "P1 split: video-id
# (proxy)", and produced macro-F1 0.256 / mean-class 0.281 / T=0.58 - numbers that look
# like a large improvement over the actor-disjoint 0.128 / 0.156 / 0.63 and are in fact
# the leak this project spent a session removing. Two hours of Blackwell time, and the
# output is unquotable.
#
# The split is the protocol of record (results/evaluation.md), so producing the other one
# has to be a decision, not an accident. Set BS_ALLOW_VIDEO_SPLIT=1 to override.
import os
if SPLIT_MODE.startswith("video-id") and not os.environ.get("BS_ALLOW_VIDEO_SPLIT"):
    raise AssertionError(
        "P1 would run on the VIDEO-ID split, which leaks actors between train and val "
        "(~30 videos per actor, so nearly every person appears on both sides). "
        "`Charades_v1_train.csv` was not found under /kaggle/input.\n\n"
        "  FIX: attach the `charades-480p` dataset - it carries the CSV.\n"
        "  Every activity number produced without it is inflated and must not be quoted "
        "next to the actor-disjoint tables in results/evaluation.md.\n"
        "  To measure the leaky protocol deliberately, set BS_ALLOW_VIDEO_SPLIT=1."
    )
print(f"P1 split: {SPLIT_MODE}")
_, val_idx = split_by_subject(split_subjects, val_frac=0.2, seed=0)
print(f"P1 val: {len(val_idx)} windows, {len(set(split_subjects[val_idx]))} subjects")
assert not (set(split_subjects[val_idx]) & set(np.delete(split_subjects, val_idx))), \
    "subject leakage between train and val"

clf = EnsembleClassifier.from_run_dir(run_dir(), device="cuda")
X = np.stack([normalise(ds.skeletons[i].astype(np.float32)) for i in val_idx])
y = ds.labels[val_idx]
assert X.shape[1:] == (30, 2, 17, 3), f"unexpected eval window layout {X.shape}"

# MIN_SUPPORT mirrors train_adl.py: below ~50 val windows a class's F1 is one prediction
# wide, so it is reported but not averaged. This matters concretely here - the real
# extraction (7,985 videos) yields standing 342 windows (0.21%) and bending_reaching 69
# (0.04%), because Charades calls those "Putting a box somewhere" / "Taking a bag from
# somewhere" and no keyword rule matches, so they land in other_idle (39.4%). Their F1 is
# ~0 and over 18 present classes that alone costs ~6 macro-F1 points. A single macro-F1
# number would make the ensemble look worse than it is with no way to see why, so the
# table carries BOTH and the starved classes are named underneath.

support = np.bincount(y, minlength=N_CLASSES)
starved = [(c, int(support[c])) for c in range(N_CLASSES) if 0 < support[c] < MIN_SUPPORT]
n_sup = int((support >= MIN_SUPPORT).sum())

per_stream = clf.per_stream_logits(X)
P1_ROWS = []
print(f"{'model':<16} {'top1':>6} {'mean-class':>10} {'macro-F1':>9} "
      f"{'F1>=' + str(MIN_SUPPORT):>9}")
for s, lg in per_stream.items():
    t1, mca_s, f1_s, f1_sup_s = scores(lg, y)
    P1_ROWS.append((s, float(t1), float(mca_s), float(f1_s), float(f1_sup_s)))
    print(f"{s:<16} {t1:>6.3f} {mca_s:>10.3f} {f1_s:>9.3f} {f1_sup_s:>9.3f}")
ens_logits = clf.logits(X)
t1, mca, f1, f1_sup = scores(ens_logits, y)
P1_ROWS.append(("ENSEMBLE", float(t1), float(mca), float(f1), float(f1_sup)))
print(f"{'ENSEMBLE':<16} {t1:>6.3f} {mca:>10.3f} {f1:>9.3f} {f1_sup:>9.3f}"
      "   <- headline P1")
print(f"\nmacro-F1 averages {int((support > 0).sum())} present classes; the last column "
      f"averages the {n_sup} with >= {MIN_SUPPORT} val windows.")
if starved:
    # Named, not silently dropped: the gap between the two columns is entirely these
    # classes, and it is a LABEL-MAP limitation to state in the write-up, not a model
    # result. Neither feeds a downstream Agent 3 feature, so it does not affect the
    # behaviour layer - which is the reason for reporting rather than re-extracting.
    print(f"{len(starved)} class(es) starved by the Charades label map "
          f"(< {MIN_SUPPORT} windows):")
    for c, n in starved:
        print(f"  class {c:>2} {CLASS_NAMES[c]:<22} support {n:>5}")
    print("  Cause: Charades has no verb for these (e.g. 'Putting a box somewhere'),")
    print("  so build_charades_map.py routes them to other_idle. Report both columns.")

# Persist the val logits. ~8 MB, and it converts a whole class of accuracy experiments -
# logit adjustment for the long-tailed label distribution, stream-subset selection,
# probability-vs-logit combination, per-class thresholds - from "book another 2-hour Kaggle
# session" into "run scripts/rescore_p1.py on a laptop". P1 computed exactly these arrays
# four times across four sessions and discarded them four times.
import pathlib, os
_RES = pathlib.Path(os.environ.get("BS_RESULTS_DIR", "/kaggle/working/results"))
_RES.mkdir(parents=True, exist_ok=True)
np.savez_compressed(_RES / "val_logits.npz", y=y.astype(np.int64),
                    subjects=ds.subjects[val_idx].astype("<U32"),
                    logits_ensemble=ens_logits.astype(np.float32),
                    **{f"logits_{s}": lg.astype(np.float32)
                       for s, lg in per_stream.items()})
print(f"wrote {_RES / 'val_logits.npz'} - re-score without a GPU: "
      f"PYTHONPATH=src python scripts/rescore_p1.py results/val_logits.npz")

In [ ]:
# Accuracy levers, measured. Three of the four are pure post-processing on the logits
# already computed above, so they cost seconds and no GPU; TTA costs one extra forward pass.
#
# Why each is here rather than assumed:
#  - LOGIT ADJUSTMENT: training uses effective-number balanced sampling, which removes only
#    ~145x of the ~942x head/tail ratio, so residual imbalance remains and mean-class is the
#    metric of record. tau is SWEPT rather than derived, with tau=0 as the control - the
#    right correction depends on how much imbalance the sampler left, which is not something
#    to guess.
#  - STREAM SUBSET: P1 shows the 4-stream average LOSING to bone alone on mean-class. So
#    the full average is not automatically right, and the subset question has 15 answers.
#  - TTA: mirror each window and average logits. AugmentConfig(flip_prob=0.5) means the
#    model trained on mirrored windows, so this averages two views it has seen.
from behaviorsense.eval.activity_eval import subset_scores, tau_sweep, combine, logit_adjust, class_prior
LEVERS = []
print(f"{'configuration':<34} {'top1':>6} {'mean-class':>11} {'macro-F1':>9}")
def lever(label, lg):
    s = scores(lg, y)
    LEVERS.append((label, *s))
    print(f"{label:<34} {s[0]:>6.3f} {s[1]:>11.3f} {s[2]:>9.3f}")
    return s

base = lever("ensemble, logit-avg (P1 headline)", ens_logits)

# 1. Stream subsets, both combination rules.
best_sub = None
for mode in ("logit", "prob"):
    rows = subset_scores(per_stream, y, mode=mode)
    top = rows[0]
    if best_sub is None or top[2] > best_sub[1][2]:
        best_sub = (mode, top)
    lever(f"best {mode}-avg subset: {'+'.join(top[0])}", combine(per_stream, top[0], mode))

# 2. Logit adjustment on the headline ensemble and on the best subset.
prior = class_prior(y)
sub_logits = combine(per_stream, best_sub[1][0], best_sub[0])
for label, lg in (("ensemble", ens_logits), (f"{'+'.join(best_sub[1][0])}", sub_logits)):
    sweep = tau_sweep(lg, y)
    assert abs(sweep[0][2] - scores(lg, y)[1]) < 1e-9, "tau=0 is not a no-op"
    peak = max(sweep, key=lambda r: r[2])
    lever(f"{label} + logit-adjust tau={peak[0]:.2f}", logit_adjust(lg, prior, peak[0]))

# 3. Test-time flip augmentation on the ensemble.
clf.tta = True
tta_logits = clf.logits(X)
clf.tta = False
lever("ensemble + TTA flip", tta_logits)

# 4. Second-person context - the one object signal that needs NO detector. The shards
# carry two person slots; slot 1 is non-zero exactly when the tracker held a second
# person, and interacting_with_person (F1 0.042, second-worst class) fails precisely
# because pose alone cannot say "someone else is here". OBJECT_PRIORS['person'] (+1.5
# log-odds on class 18) was written for this signal and has never received it. Applied
# via fuse_objects so serving and evaluation share one code path. NOTE: uncalibrated
# posteriors on purpose - T is fitted in the NEXT cell, and log-odds offsets commute with
# argmax under any temperature, so ordering does not change the decision rule.
from behaviorsense.eval.activity_eval import second_person_present
from behaviorsense.agents.activity import ActivityAgent, ActivityConfig, softmax
_second = second_person_present(ds.skeletons[val_idx])
print(f"second person visible in {int(_second.sum()):,}/{len(_second):,} val windows "
      f"({_second.mean():.1%}); class 18 support {int((y == 18).sum()):,}")
_agent = ActivityAgent(ActivityConfig())
_objs = [("person",) if s else () for s in _second]
fused = np.log(np.clip(_agent.fuse_objects(softmax(ens_logits), _objs), 1e-12, None))
lever("ensemble + second-person context", fused)
adj_fused = np.log(np.clip(_agent.fuse_objects(
    softmax(logit_adjust(ens_logits, prior, 0.25)), _objs), 1e-12, None))
lever("ens + logit-adjust 0.25 + 2nd-person", adj_fused)
# Recall/precision on class 18 specifically: a +1.5 log-odds prior helps only if the
# second-person windows are actually where class 18 lives. Print the confusion so the
# lever is diagnosable, not just a delta in an average.
for label, lg in (("without", logit_adjust(ens_logits, prior, 0.25)), ("with", adj_fused)):
    p18 = lg.argmax(1) == 18
    tp = int((p18 & (y == 18)).sum())
    print(f"  class 18 {label} context: predicted {int(p18.sum()):>5}, correct {tp:>4}, "
          f"recall {tp / max(1, int((y == 18).sum())):.3f}")

best = max(LEVERS, key=lambda r: r[2])
print()
print(f"best mean-class: {best[0]} at {best[2]:.3f} "
      f"({best[2] - base[1]:+.3f} vs the P1 headline)")
print("Selected on the validation split, so report as validation-selected, not held-out.")

In [ ]:
# Calibration: fit the temperature on these val logits. The fitted T goes into
# ActivityConfig(temperature=...) at serving time - Viterbi and abstention both consume
# probabilities, so they must mean something first.
from behaviorsense.agents.activity import fit_temperature, softmax
T = fit_temperature(ens_logits, y)
conf = softmax(ens_logits / T).max(1).mean()
acc = (ens_logits.argmax(1) == y).mean()
print(f"fitted temperature T={T:.2f}; mean confidence {conf:.3f} vs accuracy {acc:.3f}")
print(f"-> set ActivityConfig(temperature={T:.2f}) in deployment")

In [ ]:
# SEGMENT-level accuracy under Viterbi smoothing - the metric Agent 3 actually consumes.
#
# Every number above is per-WINDOW. Agent 3 never sees a window: it sees segments, built by
# smoothing the posterior sequence with a transition prior, and derives every daily feature
# from segment durations. So per-window accuracy is not the deployment metric.
#
# The first time this was measured it found a REGRESSION: under the hand-set prior
# (self_transition=0.90, uniform leakage) smoothing moved top-1 +0.011 and mean-class
# -0.040. A dominant 39% other_idle plus a sticky self-transition swallows short rare-class
# runs into the surrounding majority - it flatters top-1 and destroys tail recall, the same
# head/tail trade logit averaging makes. Two fixes are measured here against that baseline:
#   1. FIT the transition matrix from TRAINING label sequences instead of asserting it. The
#      structural prior was written when no labelled sequence data existed; there are now
#      165k windows in temporal order. Fitted on train only, evaluated on val.
#   2. Smooth the LOGIT-ADJUSTED posterior rather than the raw one. Smoothing the
#      configuration that already collapses onto the head class is the worst case for it.
# The emergency floor into `falling` is re-applied to any fitted matrix: falls are 0.5% of
# windows, so a counted matrix makes the state nearly unreachable and one-window falls are
# smoothed out of existence.
import numpy as np
from behaviorsense.agents.activity import (ActivityConfig, FALLING, build_transition_matrix,
                                           viterbi)
from behaviorsense.eval.activity_eval import (apply_emergency_floor, fit_transition_matrix,
                                              fragmentation, sequences_by_subject)
cfg_sm = ActivityConfig(temperature=float(T))
train_idx, _ = split_by_subject(ds.subjects, val_frac=0.2, seed=0)
A_fit = apply_emergency_floor(
    fit_transition_matrix(sequences_by_subject(ds.subjects[train_idx], ds.labels[train_idx])),
    FALLING, cfg_sm.emergency_floor)
log_pi = np.log(np.full(N_CLASSES, 1.0 / N_CLASSES))
subj = ds.subjects[val_idx]
y_seqs = sequences_by_subject(subj, y)
n_seq = len(y_seqs); n_win = sum(len(s) for s in y_seqs)
print(f"{n_seq} sequences, {n_win} windows (sequences of >=3 windows only)")
print(f"fitted prior: mean self-transition {np.mean(np.diag(A_fit)):.3f} "
      f"vs hand-set {cfg_sm.self_transition:.2f}")

_prior = class_prior(y)
_tau = max(tau_sweep(ens_logits, y), key=lambda r: r[2])[0]
SEGMENT_ROWS = []
def seg_eval(label, logits, A):
    post = softmax(logits / float(T))
    seqs = sequences_by_subject(subj, post)
    hit = 0; pc = np.zeros((N_CLASSES, 2), dtype=np.int64); preds = []
    for sy, sp in zip(y_seqs, seqs):
        pred = sp.argmax(1) if A is None else viterbi(
            np.log(np.clip(sp, 1e-12, None)), np.log(A), log_pi)
        preds.append(pred)
        hit += int((pred == sy).sum())
        for c, p in zip(sy, pred):
            pc[c, 0] += int(p == c); pc[c, 1] += 1
    present = pc[:, 1] > 0
    mca_ = float(np.mean(pc[present, 0] / pc[present, 1]))
    frag = fragmentation(preds, y_seqs)
    SEGMENT_ROWS.append((label, hit / n_win, mca_, frag["ratio"]))
    print(f"{label:<40} {hit / n_win:>6.3f} {mca_:>11.3f} {frag['ratio']:>9.2f}")

# Accuracy alone cannot decide argmax vs Viterbi. Agent 3 derives walking_bouts and
# mean_bout_duration_s from segment COUNTS, so a decoding that shatters one true stretch into
# nine flickering ones reports nine bouts while scoring identically per window. The
# fragmentation ratio (predicted segments / true segments, 1.0 ideal) is what smoothing is
# actually for, and without it "argmax wins on mean-class" is half an argument.
print(f"{'decoding':<40} {'top1':>6} {'mean-class':>11} {'frag':>9}")
A_hand = build_transition_matrix(cfg_sm)
adj = logit_adjust(ens_logits, _prior, _tau)
seg_eval("per-window argmax", ens_logits, None)
seg_eval("Viterbi, hand-set prior", ens_logits, A_hand)
seg_eval("Viterbi, fitted prior", ens_logits, A_fit)
seg_eval(f"argmax, logit-adjust tau={_tau:.2f}", adj, None)
seg_eval(f"Viterbi fitted + logit-adjust tau={_tau:.2f}", adj, A_fit)
_best = max(SEGMENT_ROWS, key=lambda r: r[2])
print()
print(f"best mean-class: {_best[0]} ({_best[2]:.3f}, "
      f"{_best[2] - SEGMENT_ROWS[0][2]:+.3f} vs per-window argmax) at fragmentation "
      f"{_best[3]:.2f}x")
print("Pick on BOTH columns: mean-class is label quality, frag is whether the segment")
print("structure Agent 3 counts bouts from survives. Neither alone decides it.")

In [ ]:
# P2 - leave-one-dataset-out on the fall sources: train-side generalisation is fixed
# (the ensemble saw only Charades ADL + the other fall sources via train_fall), so this
# measures how the FALL signal transfers to an unseen recording setup.
#
# TWO MODELS ARE SCORED, because this cell used to score only the first and the write-up
# then attributed its numbers to the second. The ADL ensemble's classes 7+8 are a fall
# signal, but `runs/fall/best.pt` is the DEDICATED binary head - the one that reports
# AUPRC 0.822 and owns the 0.951-sensitivity operating point - and it was never evaluated
# leave-one-dataset-out at all. Its 0.822 comes from train_fall.py's own internal val
# split, so "the fall head does not transfer" was a claim about a different model.
import glob, numpy as np, torch
from behaviorsense.data.skeleton_dataset import SkeletonWindowDataset, normalise
from behaviorsense.models.ensemble import windows_to_tensor
from behaviorsense.models.stgcnpp import STGCNpp, make_stream
FALL = shard_paths("fall")
assert FALL, f"no fall shards found. Attached: {ATTACHED}"
fds = SkeletonWindowDataset([*map(str, FALL)])
sources = np.asarray(fds.datasets)
FALL_CLASSES = (7, 8)
is_fall = np.isin(fds.labels, FALL_CLASSES)

def auroc(score, y):
    if not (y.any() and (~y).any()):
        return float("nan")
    order = np.argsort(score)
    ranks = np.empty(len(score)); ranks[order] = np.arange(1, len(score) + 1)
    return (ranks[y].sum() - y.sum() * (y.sum() + 1) / 2) / (y.sum() * (~y).sum())

# Load the dedicated fall head if it is present. Same stream the trainer used, read from
# the checkpoint rather than assumed - a wrong stream loads silently and scores nonsense.
P2_DEVICE = getattr(clf, "device", "cuda")   # follows the ensemble; harness patches it
_fh = sorted(pathlib.Path(run_dir()).glob("fall/best.pt"))
fall_head, fall_stream = None, None
if _fh:
    _ck = torch.load(str(_fh[0]), map_location="cpu", weights_only=False)
    fall_stream = (_ck.get("args") or {}).get("stream", "joint")
    _m = STGCNpp(n_classes=1)
    _state = _ck.get("ema") or _ck["model"]
    _m.load_state_dict({k[len("net."):] if k.startswith("net.") else k: v
                        for k, v in _state.items()}, strict=True)
    fall_head = _m.eval().to(P2_DEVICE)
    print(f"fall head loaded from {_fh[0].parent.name}/best.pt, stream={fall_stream}, "
          f"recorded best AUPRC {_ck.get('best', float('nan')):.3f}")
else:
    print("no runs/fall/best.pt under the mounted runs - ADL-ensemble column only")

P2_ROWS = []
print(f"{'held-out':<12} {'n':>6} {'fall%':>6} {'ADL-ens AUROC':>14} {'FALL-HEAD AUROC':>16}")
for held in sorted(set(sources)):
    idx = np.where(sources == held)[0]
    # SHARD layout [N,T,M,17,3]. fds[i][0] returns [C,T,V,M] - already permuted for the
    # model - and logits() rejects it. P1 was fixed for exactly this and P2 was left
    # behind, so it died after 11 minutes with "expected [N,T,M,17,3], got (1159,3,30,17,2)".
    Xh = np.stack([normalise(fds.skeletons[i].astype(np.float32)) for i in idx])
    yh = is_fall[idx]
    a_ens = auroc(softmax(clf.logits(Xh))[:, list(FALL_CLASSES)].sum(1), yh)
    a_head = float("nan")
    if fall_head is not None:
        with torch.no_grad():
            t = windows_to_tensor(Xh)
            parts = [torch.sigmoid(fall_head(make_stream(t[s:s + 64].to(P2_DEVICE),
                                                         fall_stream))).cpu().numpy()
                     for s in range(0, len(t), 64)]
        # FallHead.forward squeezes the single logit; we load the bare STGCNpp, which
        # returns [N,1]. ravel() rather than squeeze() so a 1-window fold cannot
        # collapse to a scalar and silently break the ranking.
        a_head = auroc(np.concatenate(parts).ravel(), yh)
    P2_ROWS.append((str(held), int(len(idx)), float(yh.mean()), float(a_ens), float(a_head)))
    print(f"{held:<12} {len(idx):>6} {yh.mean():>6.1%} {a_ens:>14.3f} {a_head:>16.3f}")
print()
print("The FALL-HEAD column is the one the 0.822 AUPRC / 0.951-sensitivity claims belong to.")
print("If it is far below its own validation AUROC, that operating point is in-domain only.")

In [ ]:
# Ablation: logit vs probability combination (product-of-experts vs mixture).
alt = EnsembleClassifier.from_run_dir(run_dir(),
                                      device="cuda", combine="prob")
# scores() returns FOUR values since the F1>=MIN_SUPPORT column was added; this call
# site still unpacked three and died with "too many values to unpack" after P1 and P2
# had already run. Unpack all four here too.
t1a, mcaa, f1a, f1_supa = scores(alt.logits(X), y)
ABLATION_ROWS = [("logit-average", float(t1), float(mca), float(f1), float(f1_sup)),
                 ("prob-average", float(t1a), float(mcaa), float(f1a), float(f1_supa))]
print(f"{'combination':<16} {'top1':>6} {'mean-class':>10} {'macro-F1':>9} "
      f"{'F1>=' + str(MIN_SUPPORT):>9}")
print(f"{'logit-average':<16} {t1:>6.3f} {mca:>10.3f} {f1:>9.3f} {f1_sup:>9.3f}"
      "   <- headline")
print(f"{'prob-average':<16} {t1a:>6.3f} {mcaa:>10.3f} {f1a:>9.3f} {f1_supa:>9.3f}")
print()
print("Logit averaging is a product of experts, probability averaging a mixture. The")
print("headline uses logits; this row is the ablation that justifies that choice rather")
print("than asserting it.")

In [ ]:
# Hallucination benchmark with the REAL model. The Kaggle Model mount path varies -
# find it, then run all three arms. Stub anchors (already in results/hallucination.md) are
# what make these numbers interpretable.
#
# Run 1 produced 'unconstrained: 245 claims emitted, 0 scorable, nan%' and 39.9% for the
# constrained arm. Both were artefacts of the prompt never naming the output fields, so the
# free model could not guess the envelope and the constrained model mis-assigned values.
# With the field mapping stated, run 2 gave 8.2% / 6.5% with zero schema rejections.
#
# Runs 2 and 3 then disagreed. Free was bit-identical (509 claims, 476 faithful, twice) but
# constrained moved 498/457 -> 504/452, flipping the verdict from p=0.29 to p=0.03. A code
# audit of the two paths side by side - which is what should have happened after run 1 -
# found THREE ways they differed other than the grammar:
#   1. free passed do_sample=False; constrained passed only max_new_tokens to outlines and
#      inherited Qwen's generation_config (do_sample=True, temperature=0.7)
#   2. free applied the chat template; constrained handed outlines the raw prompt, so one
#      arm got a Qwen chat turn and the other a naked instruction block
#   3. repetition_penalty=1.05 from Qwen's config penalised exactly the verbatim copying
#      this task is scored on (both arms equally, so not a confound - but it inflates the
#      absolute rate)
# force_greedy() + _chat_text() + a cached output type fix all three. Test R14 now audits
# every axis on which the arms could differ, on CPU, so this class of defect costs seconds
# instead of a 2-hour session. The cached guide should also cut the constrained arm's
# 49.0 s/report substantially - watch the progress lines.
#
# This is the first decoding-matched run. Earlier constrained numbers should not be quoted.
#
# --dump writes every claim plus its verdict to JSONL. Run 2 could not answer "which values
# were misquoted, and by how much" because it kept only aggregates, and answering it cost a
# second 2-hour session. Now any re-analysis is a CPU rescore of that file.
import glob, subprocess, sys, os
qwen = sorted(glob.glob("/kaggle/input/**/config.json", recursive=True))
qwen = [p for p in qwen if "qwen" in p.lower()]
assert qwen, "attach the Qwen2.5-7B-Instruct Kaggle Model as an input"
QWEN_PATH = os.path.dirname(qwen[0])
print("Qwen at:", QWEN_PATH)
subprocess.run(
    [sys.executable, str(SCRIPTS / "eval_hallucination.py"),
     "--backend", "qwen", "--model-path", QWEN_PATH, "--days", "60",
     "--report", "/kaggle/working/results/hallucination_qwen.md",
     "--dump", "/kaggle/working/results/hallucination_qwen_claims.jsonl"],
    env={**os.environ, "PYTHONPATH": str(SRC)},
    check=True)

In [ ]:
# Write every table this session produced, then list what is in results/.
#
# The first full run printed P1, the per-class breakdown, calibration, P2 and the ablation
# to the log and wrote NOTHING - the bundler only globbed for *.md, and the single file
# present was hallucination_qwen.md from the subprocess. Nine hours of GPU output existed
# only as scrollback. These tables are the dissertation numbers, so they are written from
# the in-memory rows collected above.
import pathlib, os
RES = pathlib.Path(os.environ.get("BS_RESULTS_DIR", "/kaggle/working/results"))
RES.mkdir(parents=True, exist_ok=True)

def table(header, rows, fmt):
    out = ["| " + " | ".join(header) + " |", "|" + "|".join(["---"] * len(header)) + "|"]
    out += ["| " + " | ".join(fmt(r)) + " |" for r in rows]
    return out

L = ["# P1 - subject-disjoint validation", "",
     f"- {len(y)} val windows, {len(set(ds.subjects[val_idx]))} subjects, "
     f"split seed 0 (same as training)",
     f"- macro-F1 averages {int((support > 0).sum())} present classes; the last column "
     f"averages the {n_sup} with >= {MIN_SUPPORT} windows", ""]
L += table(["model", "top1", "mean-class", "macro-F1", f"F1>={MIN_SUPPORT}"], P1_ROWS,
           lambda r: [f"`{r[0]}`" if r[0] != "ENSEMBLE" else "**ENSEMBLE**",
                      f"{r[1]:.3f}", f"{r[2]:.3f}", f"{r[3]:.3f}", f"{r[4]:.3f}"])
# State the finding that contradicts the usual expectation instead of leaving a reader to
# infer it from the table: the ensemble wins top-1 and LOSES mean-class.
best_mca = max(P1_ROWS[:-1], key=lambda r: r[2])
if P1_ROWS[-1][2] < best_mca[2]:
    L += ["", f"**The ensemble improves top-1 but loses mean-class accuracy versus "
              f"`{best_mca[0]}` alone ({P1_ROWS[-1][2]:.3f} vs {best_mca[2]:.3f}).** "
              f"Logit averaging is a product of experts, so a stream that is confidently "
              f"wrong on a rare class can veto it; on a long-tailed label distribution "
              f"that trades tail recall for head accuracy. `{best_mca[0]}` is therefore "
              f"the better deployment checkpoint on the mean-class criterion."]
if starved:
    L += ["", f"Starved by the Charades label map (< {MIN_SUPPORT} windows): "
              + ", ".join(f"`{CLASS_NAMES[c]}` ({n})" for c, n in starved)
              + ". A label-map limitation, not a model result."]
L += ["", "## Calibration", "",
      f"- fitted temperature **T={T:.2f}**; mean confidence {conf:.3f} vs accuracy {acc:.3f}",
      f"- set `ActivityConfig(temperature={T:.2f})` in deployment", ""]
L += ["## P2 - leave-one-dataset-out (fall sources)", ""]
L += table(["held-out", "n", "fall%", "ADL-ens AUROC", "fall-head AUROC"], P2_ROWS,
           lambda r: [f"`{r[0]}`", str(r[1]), f"{r[2]:.1%}",
                      "n/a" if r[3] != r[3] else f"{r[3]:.3f}",
                      "n/a" if r[4] != r[4] else f"{r[4]:.3f}"])
L += ["",
      "Two models, deliberately. The ADL ensemble's classes 7+8 are a fall signal; "
      "`runs/fall/best.pt` is the dedicated binary head that owns the AUPRC 0.822 and the "
      "0.951-sensitivity operating point. Only the second column speaks to whether those "
      "claims transfer - the cell used to score the first and the write-up attributed its "
      "numbers to the second.", ""]
L += ["", "## Accuracy levers (validation-selected)", ""]
L += table(["configuration", "top1", "mean-class", "macro-F1"], LEVERS,
           lambda r: [f"`{r[0]}`", f"{r[1]:.3f}", f"{r[2]:.3f}", f"{r[3]:.3f}"])
_best = max(LEVERS, key=lambda r: r[2])
L += ["",
      f"Best mean-class: `{_best[0]}` at {_best[2]:.3f}, {_best[2] - LEVERS[0][2]:+.3f} "
      f"against the P1 headline. Logit adjustment and subset selection are decision-rule "
      f"changes on the same weights; TTA is one extra forward pass. None needs retraining. "
      f"All were selected on this validation split, so they are validation-selected numbers "
      f"and must be reported as such.", ""]
L += ["## Segment-level accuracy (what Agent 3 consumes)", ""]
L += table(["decoding", "top1", "mean-class", "frag ratio"], SEGMENT_ROWS,
           lambda r: [f"`{r[0]}`", f"{r[1]:.3f}", f"{r[2]:.3f}", f"{r[3]:.2f}"])
L += ["",
      f"Fitted transition prior: mean self-transition {np.mean(np.diag(A_fit)):.3f} against "
      f"the hand-set {cfg_sm.self_transition:.2f}. `frag ratio` is predicted segments over "
      f"true segments (1.00 ideal): mean-class is label quality, frag is whether the segment "
      f"structure Agent 3 counts bouts from survives. Neither column alone decides the "
      f"deployment choice.", ""]
L += ["## Ablation - logit vs probability combination", ""]
L += table(["combination", "top1", "mean-class", "macro-F1", f"F1>={MIN_SUPPORT}"],
           ABLATION_ROWS,
           lambda r: [f"`{r[0]}`", f"{r[1]:.3f}", f"{r[2]:.3f}", f"{r[3]:.3f}",
                      f"{r[4]:.3f}"])
L += ["", "Logit averaging is a product of experts, probability averaging a mixture. The",
      "headline uses logits; this row is the ablation that justifies that choice."]
(RES / "evaluation.md").write_text("\n".join(L) + "\n", encoding="utf-8")

for f in sorted(RES.glob("*.md")):
    print(f"- {f.name} ({f.stat().st_size} bytes)")
print()
print("Save Version, then create/update dataset behaviorsense-results from the output -")
print("these tables are the dissertation numbers and the session log is not a record.")